# Load KPI dataset and filter chosen Case

## Step 1 - Loading the stacked KPI dataset and filtering Case Study D

In [74]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [75]:
# Location of the preliminary training data (all KPIs combined)
train_file_path = "../data/KPI-Anomaly-Detection-master/Preliminary_dataset/train.csv"

# Load the full training table
kpi_train_full = pd.read_csv(train_file_path)

# KPI selected for Case Study D
case_d_kpi_id = "da403e4e3f87c9e0"

# Keep only rows that belong to the selected KPI
df = kpi_train_full[kpi_train_full["KPI ID"] == case_d_kpi_id].copy()

df = df.sort_values("timestamp").reset_index(drop=True)

df.head()


,timestamp,value,label,KPI ID
0,1493568000,1.666667,0,da403e4e3f87c9e0
1,1493568060,1.833333,0,da403e4e3f87c9e0
2,1493568120,1.750000,0,da403e4e3f87c9e0
3,1493568180,2.000000,0,da403e4e3f87c9e0
4,1493568240,1.916667,0,da403e4e3f87c9e0


In [76]:
# Create a readable datetime column from the Unix timestamp
df["time"] = pd.to_datetime(df["timestamp"], unit="s")

# Rename label -> is_anomaly for cross-case consistency
df = df.rename(columns={"label": "is_anomaly"})

# Arrange columns so the time information appears first
df = df[["time", "timestamp", "value", "is_anomaly", "KPI ID"]]

df.head()

,time,timestamp,value,is_anomaly,KPI ID
0,2017-04-30 16:00:00,1493568000,1.666667,0,da403e4e3f87c9e0
1,2017-04-30 16:01:00,1493568060,1.833333,0,da403e4e3f87c9e0
2,2017-04-30 16:02:00,1493568120,1.750000,0,da403e4e3f87c9e0
3,2017-04-30 16:03:00,1493568180,2.000000,0,da403e4e3f87c9e0
4,2017-04-30 16:04:00,1493568240,1.916667,0,da403e4e3f87c9e0


### Step 2 - Keep only standardised core columns (drop Unix timestamp + KPI ID from the working df)


In [77]:
# New working dataframe creation 
df = df[["time", "value", "is_anomaly"]].copy()

# Add case study identifier
df["case_study"] = "aiops_kpi"

# enforce clean types
df["is_anomaly"] = df["is_anomaly"].astype(int)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

df.head()


,time,value,is_anomaly,case_study
0,2017-04-30 16:00:00,1.666667,0,aiops_kpi
1,2017-04-30 16:01:00,1.833333,0,aiops_kpi
2,2017-04-30 16:02:00,1.750000,0,aiops_kpi
3,2017-04-30 16:03:00,2.000000,0,aiops_kpi
4,2017-04-30 16:04:00,1.916667,0,aiops_kpi


###  Quick verification

In [78]:
df[["time","value","is_anomaly","case_study"]].dtypes

time          datetime64[ns]
value                float64
is_anomaly             int64
case_study            object
dtype: object


In step 1 and 2, the KPI dataframe is aligned with the common schema used across all case studies. The `timestamp` column is converted to a proper datetime type, the rows are sorted chronologically, and the column is renamed to `time`. A `case_study` column with the value `"aiops_kpi"` is added so that this dataset can later be stacked or filtered alongside the other case studies using the same core columns (`time`, `value`, `is_anomaly`, `case_study`).


## Step 3 - Missing values and Duplicates rows sanity check

### A) Missing values per column

In [79]:
# Count missing values in each column
missing_counts = df.isna().sum()

# Percentage of missing values in each column
missing_percentages = (df.isna().mean() * 100).round(3)

# Combine into a summary table
missing_summary = (
    pd.DataFrame(
        {
            "Column": missing_counts.index,
            "Missing count": missing_counts.values,
            "Missing (%)": missing_percentages.values,
        }
    )
    .sort_values("Missing count", ascending=False)
    .reset_index(drop=True)
)

missing_summary

,Column,Missing count,Missing (%)
0,time,0,0.0
1,value,0,0.0
2,is_anomaly,0,0.0
3,case_study,0,0.0


### B) Duplicate rows summary 

In [80]:
# Duplicate rows summary
duplicate_count = df.duplicated().sum()
total_rows = len(df)

duplicate_summary = pd.DataFrame(
    [
        ("Total rows", total_rows),
        ("Number of duplicate rows", duplicate_count),
        (
            "Duplicate rows (%)",
            round(duplicate_count / total_rows * 100, 3),
        ),
    ],
    columns=["Statistic", "Value"],
)

duplicate_summary


,Statistic,Value
0,Total rows,129035.0
1,Number of duplicate rows,0.0
2,Duplicate rows (%),0.0



The missing-values table confirms that the AIOps KPI series has no missing entries in any of the core columns (`time`, `value`, `is_anomaly`, `case_study`). The duplicate-rows summary also reports zero exact duplicate records across the dataset. These checks indicate that the filtered KPI time series is clean at the core-schema level, so no imputation or de-duplication is required before proceeding to time-feature construction and chronological split design. Recording these results explicitly supports transparency and reproducibility in the preprocessing pipeline.


## Step 4 - Create basic time-based features

In [81]:
# Hour of day (0–23)
df["hour_of_day"] = df["time"].dt.hour

# Day of week (0 = Monday, 6 = Sunday)
df["day_of_week"] = df["time"].dt.dayofweek

# Weekend flag (Saturday = 5, Sunday = 6)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Quick check
df[["time", "hour_of_day", "day_of_week", "is_weekend"]].head()


,time,hour_of_day,day_of_week,is_weekend
0,2017-04-30 16:00:00,16,6,1
1,2017-04-30 16:01:00,16,6,1
2,2017-04-30 16:02:00,16,6,1
3,2017-04-30 16:03:00,16,6,1
4,2017-04-30 16:04:00,16,6,1


This step derives simple calendar features from the `time` column to provide
additional structure for later analysis and modelling. The timestamp is broken
down into:

- `hour_of_day`: the hour of the day (0–23),
- `day_of_week`: the day index (0 = Monday, …, 6 = Sunday),
- `is_weekend`: an indicator variable that is 1 for Saturday and Sunday, and 0
  for weekdays.

These features do not change the underlying temperature values, but they allow
later models and visualisations to capture patterns that depend on time of day
or weekday/weekend effects.


## Step 5 - Inspect time span and anomaly distribution

### 5A) Class Balance 

In [82]:
# Total number of rows in the KPI series
total_rows = len(df)

# Total number of labelled anomalies (1s)
anomaly_rows = int(df["is_anomaly"].sum())

# Compute proportions for reporting
normal_rows = total_rows - anomaly_rows
normal_pct = round((normal_rows / total_rows) * 100, 2)
anomaly_pct = round((anomaly_rows / total_rows) * 100, 2)

# Build a class-balance table 
class_balance = pd.DataFrame(
    {
        "Class": ["Normal (0)", "Anomaly (1)"],
        "Count": [normal_rows, anomaly_rows],
        "Proportion (%)": [normal_pct, anomaly_pct],
    }
)

class_balance


,Class,Count,Proportion (%)
0,Normal (0),121369,94.06
1,Anomaly (1),7666,5.94


- **5A - Class balance (label density)**
  - The dataset is labelled and imbalanced, with anomalies forming a minority class (see the Step 5A class-balance table).
  - This supports using the KPI as the “higher-anomaly” case in the portfolio relative to the NAB series and motivates later evaluation choices that remain meaningful under class imbalance.

### 5B) Anomaly preview table 

In [83]:
# Filter only labelled anomalies to verify time alignment and inspect values
anomaly_df = df.loc[df["is_anomaly"] == 1, ["time", "value"]].copy()

# Add a simple ID for readability in the preview (row index values are original row indicies from df, ie. reset index=False)
anomaly_df = anomaly_df.reset_index(drop=False).rename(columns={"index": "row index"})

# Sort chronologically to match the time-series narrative
anomaly_df = anomaly_df.sort_values("time").reset_index(drop=True)

# Show a small preview 
anomaly_preview = anomaly_df.head(15)

anomaly_preview


,row index,time,value
0,5107,2017-05-04 06:09:00,4.750000
1,5108,2017-05-04 06:10:00,6.666667
2,5109,2017-05-04 06:11:00,5.583333
3,5110,2017-05-04 06:12:00,5.750000
4,5111,2017-05-04 06:13:00,6.333333
5,5112,2017-05-04 06:14:00,5.666667
6,5113,2017-05-04 06:15:00,6.666667
7,5114,2017-05-04 06:16:00,6.500000
8,5115,2017-05-04 06:17:00,7.083333
9,5116,2017-05-04 06:18:00,6.333333


- **5B - Anomaly preview (time alignment and incident-like bursts)**
  - The anomaly preview confirms that labels are correctly aligned to the time axis.
  - The earliest anomalies cluster tightly in time (multiple consecutive anomaly points around **2017-05-04 ~06:09–06:22**, plus another anomaly later that morning), indicating an incident-like burst rather than isolated single-point noise.
  - This supports the narrative that the KPI contains bursts of anomalous activity consistent with operational incidents.


### 5C) Dataset time span summary (start/end coverage and duration)


In [84]:
# Find the earliest and the latest timestamp in the full series (dataset start and dataset end)
start_time = df["time"].min()
end_time = df["time"].max()

# Convert the time span into days:
# - total_seconds() gives the difference in seconds
# - converts seconds -> days (60s/min * 60min/hr * 24hr/day)
duration_days = round((end_time - start_time).total_seconds() / (60 * 60 * 24), 3)

# Summary table for reporting and later split design
time_span_summary = pd.DataFrame(
    {
        "Start time": [start_time],
        "End time": [end_time],
        "Rows": [len(df)],
        "Duration days": [duration_days],
    }
)

time_span_summary


,Start time,End time,Rows,Duration days
0,2017-04-30 16:00:00,2017-07-31 04:29:00,129035,91.52


- **5C - Time coverage and scale**
  - **Start time:** 2017-04-30 16:00:00  
  - **End time:** 2017-07-31 04:29:00  
  - **Rows:** 129,035  
  - **Duration:** 91.52 days  
  - The series spans roughly three months, providing enough time depth for splits that represent pre-incident, incident, and post-incident behaviour and for drift/regime analysis.

### 5D) Dominant sampling interval (in seconds)

In [85]:
# Compute time differences between consecutive rows to find the sampling interval
time_differences_sec = df["time"].diff().dt.total_seconds()

# Count how often each interval occurs (drop the first NaN delta)
sampling_interval_counts = (
    time_differences_sec.dropna()
    .value_counts()
    .to_frame(name="Count")
    .reset_index(names="Differences in seconds")
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

# Show the top 10 most common intervals (expected dominant interval is 60 seconds)
sampling_interval_counts.head(10)


,Differences in seconds,Count
0,60.0,128994
1,180.0,10
2,120.0,10
3,3660.0,8
4,240.0,7
5,7260.0,2
6,12120.0,1
7,106500.0,1
8,600.0,1


- **5D - Dominant sampling interval (native resolution confirmation)**
  - The dominant interval is **60 seconds** with **128,994** occurrences, confirming a 1-minute monitoring stream.
  - Rare deviations from 60 seconds occur (e.g., **120s, 180s, 240s**, and several much longer intervals), indicating occasional missing periods rather than a different base sampling rate.
  - This supports preserving the native 1-minute resolution and treating longer intervals as gaps.


### 5E) Gap summary (intervals > 60 seconds)

In [86]:
# Mark intervals that exceed the expected and common 60-second measure (these indicate missing periods)
gap_threshold_sec = 60
interval_exceeds_60s = time_differences_sec > gap_threshold_sec

# Count how many gaps exist (how many intervals are longer than 60 seconds)
gap_count = int(interval_exceeds_60s.sum())

# Compute the percentage of intervals that are gaps (intervals = rows - 1)
total_intervals = max(len(df) - 1, 1)
gap_pct = round((gap_count / total_intervals) * 100, 3)

# Record the largest observed gap in seconds (0 if no gaps exist)
max_gap_sec = float(time_differences_sec[interval_exceeds_60s].max()) if gap_count > 0 else 0.0

# Summary table for reporting
gap_summary = pd.DataFrame(
    {
        "Common Gap threshold (seconds)": [gap_threshold_sec],
        "Number of Gaps": [gap_count],
        "Gaps (%) of Total intervals": [gap_pct],
        "Max Gap (seconds)": [max_gap_sec],
    }
)

gap_summary


,Common Gap threshold (seconds),Number of Gaps,Gaps (%) of Total intervals,Max Gap (seconds)
0,60,40,0.031,106500.0


  - Step 5 confirms that the KPI series is a long, predominantly regular 1-minute operational stream with a small number of missing periods and a clear early burst of anomalous activity.

- **5E - Gap frequency and worst-case missing period**
  - Using a gap threshold of **> 60 seconds**, the series has:
    - **Number of gaps:** 40  
    - **Gaps (% of total intervals):** 0.031%  
    - **Max gap:** 106,500 seconds  
  - Gaps are very rare (the time axis is mostly continuous), but at least one very large missing period exists (approximately 29.6 hours). This must be acknowledged during split design and any window-based modelling.
  - The combination of rare gaps and one very large gap indicates the series is operationally regular, but includes a substantial missing block that should not be mistaken for a regime change.

- **Story so far (what Step 5 establishes)**
  - The KPI behaves like a realistic operational metric: regular 1-minute cadence, long coverage window, minority anomalies, and incident-style bursts of anomalous points.
  - The data are well-suited for Case D’s role: testing methods under higher anomaly density and incident regimes while remaining aligned to the shared cross-case preprocessing framework.


### List the top time gaps with timestamps

In [87]:
# Find intervals longer than the expected 60 seconds
gap_threshold_sec = 60
time_differences_sec = df["time"].diff().dt.total_seconds()

# Build a table showing the gap start/end times and gap size
gap_events = df.loc[time_differences_sec > gap_threshold_sec, ["time"]].copy()
gap_events["Gap seconds"] = time_differences_sec[time_differences_sec > gap_threshold_sec].values

# Add the timestamp just before the gap (previous row time)
gap_events["Previous time"] = df["time"].shift(1).loc[time_differences_sec > gap_threshold_sec].values

# Order by biggest gaps first and preview the top 10
gap_events = gap_events[["Previous time", "time", "Gap seconds"]].sort_values("Gap seconds", ascending=False).reset_index(drop=True)
gap_events.head(10)


,Previous time,time,Gap seconds
0,2017-06-02 19:26:00,2017-06-04 01:01:00,106500.0
1,2017-05-17 07:39:00,2017-05-17 11:01:00,12120.0
2,2017-05-29 06:59:00,2017-05-29 09:00:00,7260.0
3,2017-05-29 02:59:00,2017-05-29 05:00:00,7260.0
4,2017-05-03 03:59:00,2017-05-03 05:00:00,3660.0
5,2017-05-05 02:59:00,2017-05-05 04:00:00,3660.0
6,2017-05-09 04:59:00,2017-05-09 06:00:00,3660.0
7,2017-05-28 20:59:00,2017-05-28 22:00:00,3660.0
8,2017-05-28 22:59:00,2017-05-29 00:00:00,3660.0
9,2017-05-17 00:59:00,2017-05-17 02:00:00,3660.0


- **Top time-gap inspection (Case D: AIOps KPI)**
  - The KPI series is expected to arrive at a **regular 60-second measure**. Any interval larger than 60 seconds indicates **missing minutes** in the time axis (a “gap”), not a change in the underlying sampling design.

- **What the gap table shows**
  - The table lists the **timestamp just before the gap**, the **timestamp when data resumes**, and the **gap duration in seconds**.
  - Most gaps are **short missing blocks** (for example ~3,660 seconds, roughly one hour), while one gap is substantially larger (**106,500 seconds**, roughly 29.6 hours).

- **Interpretation**
  - These gaps represent periods where KPI measurements are **not recorded** (for example due to monitoring/ingestion interruption, downtime, or missing backfill). The dataset is anonymised, so the exact operational cause cannot be confirmed from the data alone.
  - The presence of a few short gaps and one large gap indicates that the series is **mostly continuous**, but contains at least one **major missing block** that should be treated as a break in continuity.

- **Decision for preprocessing**
  - Gaps are **not filled** (no forward-fill or interpolation) to avoid inventing synthetic values that could distort anomaly structure and drift interpretation.
  - Instead, gaps are treated as **missing periods**, and their existence is documented explicitly for transparency.

- **Implication for modelling and split design**
  - For window-based models, gaps can create non-continuos sequences. During window creation, windows that **cross a gap** (interval > 60s inside the window) will be excluded so that model inputs represent genuinely continuos time.
  - During split design (Step 6), boundaries will be chosen to avoid being placed **inside** a large gap and to ensure that training, validation, and test segments remain interpretable and defensible.


## Step 6 - Define chronological train/validation/test boundaries

### 6A) Daily anomaly ratio

In [88]:
# Goal: produce an objective daily view of anomaly intensity to guide split boundaries

# Convert timestamps to calendar dates (daily buckets for interpretation)
df["date"] = df["time"].dt.date

# Count total points per day and anomaly points per day
daily_counts = (
    df.groupby("date")
      .agg(
          total_points=("is_anomaly", "size"),
          anomaly_points=("is_anomaly", "sum")
      )
      .reset_index()
)

# Compute anomaly ratio per day (share of points that are labelled anomalies)
daily_counts["anomaly_ratio"] = (daily_counts["anomaly_points"] / daily_counts["total_points"]).round(4)

# Sort by date to preserve chronology
daily_counts = daily_counts.sort_values("date").reset_index(drop=True)

# Preview the daily pattern (top 20 rows)
daily_counts.head(20)


,date,total_points,anomaly_points,anomaly_ratio
0,2017-04-30,480,0,0.0000
1,2017-05-01,1440,0,0.0000
2,2017-05-02,1440,0,0.0000
3,2017-05-03,1378,0,0.0000
4,2017-05-04,1435,28,0.0195
5,2017-05-05,1380,18,0.0130
6,2017-05-06,1440,41,0.0285
7,2017-05-07,1440,0,0.0000
8,2017-05-08,1440,0,0.0000
9,2017-05-09,1380,0,0.0000


- **Step 6A - Daily anomaly ratio (why this is done)**
  - Case D contains bursty anomaly behaviour, so split boundaries should be based on evidence rather than chosen arbitrarily.
  - The series is aggregated to **daily buckets** to measure how anomaly activity changes over time in a clear, examiner-friendly way.
  - For each date, the table reports:
    - `total_points`: number of KPI measurements available that day (expected ~1440 for a full 1-minute day; lower counts can reflect partial days or missing periods),
    - `anomaly_points`: number of labelled anomalies that day,
    - `anomaly_ratio`: anomaly_points / total_points, representing the share of the day that is anomalous.

- **What Step 6A reveals about the data**
  - Most days have `anomaly_ratio = 0.0000`, indicating long stretches of stable, normal behaviour.
  - Some days have non-zero anomaly ratios, showing that anomalies are concentrated in episodes rather than evenly distributed across time.
  - This supports using Case D to study incident regimes and drift under class imbalance.


### 6A.1) Top anomaly days

In [89]:
# Show the days with the highest anomaly ratios (candidate incident days)
top_anomaly_days = (
    daily_counts.sort_values("anomaly_ratio", ascending=False)
                .head(15)
                .reset_index(drop=True)
)

top_anomaly_days

,date,total_points,anomaly_points,anomaly_ratio
0,2017-06-30,1440,1440,1.0000
1,2017-07-01,1440,1440,1.0000
2,2017-07-02,1440,1440,1.0000
3,2017-06-29,1440,1440,1.0000
4,2017-06-28,1440,771,0.5354
5,2017-07-03,1440,122,0.0847
6,2017-05-20,1440,50,0.0347
7,2017-06-10,1440,47,0.0326
8,2017-05-13,1440,45,0.0312
9,2017-05-06,1440,41,0.0285



- **Step 6A.1 - Top anomaly days (how this guides split design)**
  - The daily table is then sorted by `anomaly_ratio` to identify the most anomaly-dense days (candidate incident periods).
  - The results show a clear **high-anomaly incident regime**:
    - **2017-06-29 to 2017-07-02** have `anomaly_ratio = 1.0000` (every point that day is labelled anomalous).
    - **2017-06-28** has `anomaly_ratio = 0.5354` (more than half the day anomalous).
    - Additional elevated days occur around **early July** and scattered earlier dates (lower ratios).
  - These findings provide an objective basis for Step 6 boundary selection:
    - training should prioritise earlier low-anomaly periods,
    - validation can bridge into the onset of increased anomaly activity,
    - test should include the incident regime (late June to early July) and relevant post-incident behaviour.


### 6A.2) Place the incident days on the full timeline (chronological view) 

In [90]:
# Flag days that have any anomalies (helps locate anomaly regimes chronologically)
daily_counts["has_anomaly"] = (daily_counts["anomaly_points"] > 0).astype(int)

# Keep only days with anomalies and show them in chronological order
anomaly_days_chronological = (
    daily_counts.loc[daily_counts["has_anomaly"] == 1, ["date", "total_points", "anomaly_points", "anomaly_ratio"]]
              .sort_values("date")
              .reset_index(drop=True)
)

anomaly_days_chronological.head(30)


,date,total_points,anomaly_points,anomaly_ratio
0,2017-05-04,1435,28,0.0195
1,2017-05-05,1380,18,0.0130
2,2017-05-06,1440,41,0.0285
3,2017-05-11,1438,28,0.0195
4,2017-05-12,1440,24,0.0167
5,2017-05-13,1440,45,0.0312
6,2017-05-18,1439,21,0.0146
7,2017-05-19,1440,34,0.0236
8,2017-05-20,1440,50,0.0347
9,2017-05-25,1380,27,0.0196


In [91]:
anomaly_days_chronological.tail(30)

,date,total_points,anomaly_points,anomaly_ratio
21,2017-06-19,1440,12,0.0083
22,2017-06-22,1437,20,0.0139
23,2017-06-24,1438,18,0.0125
24,2017-06-27,1440,17,0.0118
25,2017-06-28,1440,771,0.5354
26,2017-06-29,1440,1440,1.0000
27,2017-06-30,1440,1440,1.0000
28,2017-07-01,1440,1440,1.0000
29,2017-07-02,1440,1440,1.0000
30,2017-07-03,1440,122,0.0847


- **Step 6A.2 — Chronological list of anomaly days (bridging evidence to split boundaries)**
  - After identifying the most anomaly-dense days (Step 6A.1), this step lists **all days with at least one labelled anomaly** in **date order**.
  - The purpose is to locate anomaly activity on the full timeline so split boundaries can be chosen based on observed regimes rather than isolated “top-day” rankings.

- **What the chronological view shows**
  - Anomalies occur on multiple scattered days in May and June, mostly at low to moderate daily anomaly ratios.
  - A clear incident regime emerges in late June:
    - **2017-06-28** shows a sharp increase in anomaly density (`anomaly_ratio = 0.5354`).
    - **2017-06-29 to 2017-07-02** are extreme days where the anomaly ratio reaches **1.0000** (all points on those days are labelled anomalous).
  - After the incident window, anomaly activity continues into July with lower daily ratios, suggesting a post-incident period with ongoing but reduced anomaly presence.

- **Why this matters for Step 6 split design**
  - This chronological mapping provides direct evidence for defining split regimes:
    - a predominantly stable period earlier in the series,
    - an incident window spanning late June to early July,
    - and a post-incident period through July.
  - Split boundary timestamps will be selected to ensure training captures earlier normal behaviour, while validation/test capture the onset, peak, and aftermath of the incident regime.


### 6B.1) Define the incident window boundaries (date-based, no splitting yet) 

In [92]:
# Define the incident window using the anomaly-density evidence from Steps 6A–6A.2
# Start: the first high-density day (2017-06-28)
incident_start_date = pd.to_datetime("2017-06-28").date()

# End: the last day where anomaly_ratio was 1.0000 in the peak window (2017-07-02)
incident_end_date = pd.to_datetime("2017-07-02").date()

incident_start_date, incident_end_date


(datetime.date(2017, 6, 28), datetime.date(2017, 7, 2))

### 6B.2) Convert incident dates into exact timestamp boundaries

In [93]:
# Create a date column (if it already exists, this will simply overwrite consistently)
df["date"] = df["time"].dt.date

# Find the first timestamp of the incident start date present in the data
incident_start_time = df.loc[df["date"] == incident_start_date, "time"].min()

# Find the last timestamp of the incident end date present in the data
incident_end_time = df.loc[df["date"] == incident_end_date, "time"].max()

incident_start_time, incident_end_time


(Timestamp('2017-06-28 00:00:00'), Timestamp('2017-07-02 23:59:00'))

### 6B.3) Evaluate candidate split designs (train_end sensitivity)

In [113]:
# Goal: compare training cutoffs while fixing validation end to the day before the incident starts
# and keeping the incident window inside TEST.

df["date"] = df["time"].dt.date

incident_start_date = pd.to_datetime("2017-06-28").date()
validation_end_time = df.loc[df["date"] == (incident_start_date - pd.Timedelta(days=1)), "time"].max()

candidate_train_end_dates = ["2017-06-10", "2017-06-15", "2017-06-20", "2017-06-25"]

comparison_rows = []

for date_str in candidate_train_end_dates:
    train_end_date = pd.to_datetime(date_str).date()
    training_end_time = df.loc[df["date"] == train_end_date, "time"].max()

    # Skip dates that do not exist in the dataset
    if pd.isna(training_end_time):
        continue

    # Assign split labels (default is Test, then overwrite Train and Validation)
    split_labels = pd.Series("Test", index=df.index)
    split_labels.loc[df["time"] <= training_end_time] = "Train"
    split_labels.loc[(df["time"] > training_end_time) & (df["time"] <= validation_end_time)] = "Validation"

    # Summarise rows and anomaly percentages per split
    summary = (
        pd.DataFrame({"Split": split_labels, "is_anomaly": df["is_anomaly"]})
          .groupby("Split")
          .agg(Rows=("is_anomaly", "size"), Anomalies=("is_anomaly", "sum"))
    )
    summary["Anomaly (%)"] = (summary["Anomalies"] / summary["Rows"] * 100).round(3)

    # Flatten into one comparison row (assumes all three splits exist)
    comparison_rows.append(
        {
            "Train end date": str(train_end_date),
            "Train rows": int(summary.loc["Train", "Rows"]),
            "Train anomaly (%)": float(summary.loc["Train", "Anomaly (%)"]),
            "Validation rows": int(summary.loc["Validation", "Rows"]),
            "Validation anomaly (%)": float(summary.loc["Validation", "Anomaly (%)"]),
            "Test rows": int(summary.loc["Test", "Rows"]),
            "Test anomaly (%)": float(summary.loc["Test", "Anomaly (%)"]),
        }
    )

candidate_comparison = (
    pd.DataFrame(comparison_rows)
      .sort_values("Train end date")
      .reset_index(drop=True)
)

candidate_comparison


,Train end date,Train rows,Train anomaly (%),Validation rows,Validation anomaly (%),Test rows,Test anomaly (%)
0,2017-06-10,56802,0.937,24465,0.523,47768,14.667
1,2017-06-15,63999,0.880,17268,0.562,47768,14.667
2,2017-06-20,71196,0.850,10071,0.546,47768,14.667
3,2017-06-25,78387,0.820,2880,0.590,47768,14.667


- **Step 6B.3 - Evaluate candidate split designs (training-end sensitivity)**
  - This cell tests several plausible **training cut-off dates** to identify a split design that is defensible for Case D while remaining consistent with the project’s chronological split principle.
  - The design objective for Case D is to ensure that:
    - the **peak incident regime** (late June to early July) is included in **Test** for final evaluation,
    - **Validation** represents the **lead-up period** immediately before the incident (useful for tuning without being dominated by anomalies),
    - **Training** remains largely representative of baseline behaviour for learning normal patterns.

- **How the evaluation is performed**
  - The **validation end time** is fixed to the end of the day **before** the incident starts (2017-06-27), so that Test begins at the incident onset (2017-06-28).
  - Multiple candidate **Train end dates** are evaluated (end-of-day cutoffs).
  - For each candidate option, the cell computes:
    - split sizes (rows) for Train, Validation, and Test, and
    - **anomaly percentage per split** as an indicator of anomaly contamination and incident concentration.

- **What the cell achieved**
  - It produces a comparison table showing how the choice of training cut-off changes:
    - the amount of data available for training and validation, and
    - the anomaly density in each split.
  - This enables an evidence-based selection of split boundaries that supports the research aim:
    - fair comparison of diffusion-based methods and baseline models under class imbalance,
    - evaluation that includes a realistic high-anomaly incident regime in the Test split,
    - and a transparent, reproducible justification of the chosen split configuration.


## Step 7 - Assign split labels using the chosen boundaries

In [117]:
df["date"] = df["time"].dt.date

# Final chosen design:
# - Train ends at end of 2017-06-15
# - Validation ends at end of 2017-06-27
training_end_time = df.loc[df["date"] == pd.to_datetime("2017-06-15").date(), "time"].max()
validation_end_time = df.loc[df["date"] == pd.to_datetime("2017-06-27").date(), "time"].max()

# Assign splits chronologically
df["split"] = "test"
df.loc[df["time"] <= training_end_time, "split"] = "train"
df.loc[(df["time"] > training_end_time) & (df["time"] <= validation_end_time), "split"] = "validation"

# Split summary (rows + anomalies + proportions + time span)
total_rows = len(df)

split_summary = (
    df.groupby("split")
      .agg(
          Start_time=("time", "min"),
          End_time=("time", "max"),
          Rows=("time", "size"),
          Anomalies=("is_anomaly", "sum"),
      )
      .reset_index()
)

split_order = ["train", "validation", "test"]
split_summary["split"] = pd.Categorical(split_summary["split"], categories=split_order, ordered=True)
split_summary = split_summary.sort_values("split").reset_index(drop=True)

split_summary["Proportion (%)"] = (split_summary["Rows"] / total_rows * 100).round(3)

split_summary


,split,Start_time,End_time,Rows,Anomalies,Proportion (%)
0,train,2017-04-30 16:00:00,2017-06-15 23:59:00,63999,563,49.598
1,validation,2017-06-16 00:00:00,2017-06-27 23:59:00,17268,97,13.382
2,test,2017-06-28 00:00:00,2017-07-31 04:29:00,47768,7006,37.019


- **Step 7 - Assign split labels using the final chosen boundaries (Case D)**
  - This step assigns the `split` column (`train`, `validation`, `test`) using **fixed, evidence-based chronological boundaries** selected in Step 6B.3 to support a defensible evaluation regime for the AIOps KPI.
  - Boundary timestamps are computed directly from the dataset to align splits to real observed timestamps and avoid off-grid cutoffs:
    - **Training ends:** 2017-06-15 23:59:00  
    - **Validation ends:** 2017-06-27 23:59:00  
    - **Test starts:** 2017-06-28 00:00:00 (incident onset)

- **Rationale (link to research goals)**
  - The design ensures the **high-anomaly incident regime** (late June to early July) is included in the **Test** split for final evaluation, supporting robust comparison of diffusion-based methods and baseline models under realistic incident conditions.
  - Validation covers the **lead-up period** before the incident, providing a meaningful tuning window without being dominated by anomalies.
  - Training provides a large baseline period for learning normal behaviour under the dataset’s native 1-minute cadence.

- **Resulting split segments (documented summary)**
  - **Train:** 2017-04-30 16:00:00 → 2017-06-15 23:59:00  
    - Rows: 63,999 | Anomalies: 563 | Proportion: 49.598%
  - **Validation:** 2017-06-16 00:00:00 → 2017-06-27 23:59:00  
    - Rows: 17,268 | Anomalies: 97 | Proportion: 13.382%
  - **Test:** 2017-06-28 00:00:00 → 2017-07-31 04:29:00  
    - Rows: 47,768 | Anomalies: 7,006 | Proportion: 37.019%

- **Notes for downstream steps**
  - The split assignment is strictly chronological and non-destructive (no rows removed), enabling consistent scaling (train-only fit) and later window-based modelling with contiguity enforcement across time gaps.


## Step 8 - Split-specific dataframes and Integrity check

The `split` column now tags each row in the KPI AIops dataframe as belonging to
the training, validation, or test segment. In this step, the full processed
KPI AIops dataframe is kept intact, and separate dataframes for each split are
created using boolean masks on the `split` column.

A small integrity check is then used to confirm that the number of rows in
`train_df`, `validation_df`, and `test_df` adds up exactly to the number of
rows in the full dataframe. This ensures that the chronological splitting does
not accidentally discard any data and can be safely re-run without shrinking
the dataset.


### 8A)  Create split specific dataframes and verify no rows are lost

In [121]:
# Keep track of the total number of rows in the full processed KPI AIops dataframe
total_rows_full = len(df)

# Create non-destructive split-specific dataframes using boolean masks
train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

In [122]:
# Row counts in each split dataframe
n_train = len(train_df)
n_validation = len(validation_df)
n_test = len(test_df)

# Construct integrity summary to check that no rows are lost
split_integrity_summary = pd.DataFrame(
    [
        ("Full dataset rows", total_rows_full),
        ("Train rows", n_train),
        ("Validation rows", n_validation),
        ("Test rows", n_test),
        ("Train + Validation + Test", n_train + n_validation + n_test),
        ("Split sizes add up", total_rows_full == n_train + n_validation + n_test),
    ],
    columns=["Statistic", "Value"],
)

split_integrity_summary


,Statistic,Value
0,Full dataset rows,129035
1,Train rows,63999
2,Validation rows,17268
3,Test rows,47768
4,Train + Validation + Test,129035
5,Split sizes add up,True


### 8B) Split ordering and boundary sanity checks

#### Goal: confirm splits are time-sorted, non-overlapping, and respect chronological order

In [133]:
# Ensure each split is sorted by time 
train_sorted = train_df["time"].is_monotonic_increasing
validation_sorted = validation_df["time"].is_monotonic_increasing
test_sorted = test_df["time"].is_monotonic_increasing

# dataframes
train_time = train_df["time"]
validation_time = validation_df["time"]
test_time = test_df["time"]

# Boundary checks: train ends before validation starts; validation ends before test starts
boundary_integrity = pd.DataFrame(
    [
        ("Train sorted by time", train_sorted),
        ("Validation sorted by time", validation_sorted),
        ("Test sorted by time", test_sorted),
        ("Train start time", train_time.min()),
        ("Train end time", train_time.max()),
        ("Validation start time", validation_time.min()),
        ("Validation end time", validation_time.max()),
        ("Test start time", test_time.min()),
        ("Test end time", test_time.max()),
        ("Train ends before validation starts", train_time.max() < validation_time.min()),
        ("Validation ends before test starts", validation_time.max() < test_time.min()),
    ],
    columns=["Check", "Result"],
)

boundary_integrity


,Check,Result
0,Train sorted by time,True
1,Validation sorted by time,True
2,Test sorted by time,True
3,Train start time,2017-04-30 16:00:00
4,Train end time,2017-06-15 23:59:00
5,Validation start time,2017-06-16 00:00:00
6,Validation end time,2017-06-27 23:59:00
7,Test start time,2017-06-28 00:00:00
8,Test end time,2017-07-31 04:29:00
9,Train ends before validation starts,True


### Step 8B - Integrity checks for split dataframes (chronology and boundary consistency)

- **Purpose of this step**
  - Confirm that the split-specific dataframes (`train_df`, `validation_df`, `test_df`) are internally consistent and suitable for downstream preprocessing and modelling.
  - Verify that splits are strictly chronological, non-overlapping, and correctly ordered in time to prevent leakage across train/validation/test.

- **What is checked**
  - **Within-split ordering:** each split is verified to be sorted in ascending time order (`is_monotonic_increasing`).
  - **Boundary consistency:** the latest timestamp in training occurs before the earliest timestamp in validation, and the latest timestamp in validation occurs before the earliest timestamp in test.
  - **Coverage visibility:** reported start/end timestamps for each split provide a clear summary of time coverage and confirm the split design is being applied as intended.

- **Interpretation of results**
  - If all “sorted by time” checks return `True`, each split is safe for windowing and sequential modelling.
  - If the boundary checks return `True`, the split design is non-overlapping and leakage-free under the chronological assignment rules.
  - Together, these checks provide evidence that the dataset partitioning is reproducible and methodologically defensible for later evaluation.

## Step 9 - Standardise value using training split

### 9A) Fit a StandardScaler on the KPI AIops dataset training values and store parameters

####  - Fit StandardScaler on TRAIN only (prevents leakage into scaling)

In [134]:
from sklearn.preprocessing import StandardScaler

# Initialise the scaler
scaler = StandardScaler()

# Fit using only the training split values (shape must be 2D for sklearn)
scaler.fit(train_df[["value"]])

# Store parameters for documentation and reproducibility
aiops_kpi_scaler_parameters = {
    "mean": float(scaler.mean_[0]),
    "std": float(scaler.scale_[0]),
}

aiops_kpi_scaler_parameters


{'mean': 2.62024940225034, 'std': 1.0365871216890792}

### 9B) Apply scaling to full dataframe and check training behaviour

#### - This keeps all splits on the same scale without refitting (no leakage).

In [137]:
#  Apply the fitted scaler to the FULL dataframe (train/validation/test)
df["value_scaled"] = scaler.transform(df[["value"]])

# Confirm scaling behaves as expected on TRAIN (mean ~ 0, std ~ 1)
scaled_training_summary = (
    df.loc[df["split"] == "train", "value_scaled"]
      .describe()[["mean", "std"]]
      .to_frame(name="Training value_scaled")
      .rename(index={"mean": "Mean", "std": "Std. deviation"})
      .round(4)
)

scaled_training_summary


,Training value_scaled
Mean,0.0
Std. deviation,1.0


### 9C) Summary of value_scaled by split

#### - Purpose: Show how validation/test distributions differ from training (possible drift/incident effects)


In [140]:
# Summarise scaled values
split_scaled_summary = (
    df.groupby("split")["value_scaled"]
      .describe()[["mean", "std", "min", "max"]]
      .round(3)
      .reset_index()
)

# Keep split order consistent
split_order = ["train", "validation", "test"]
split_scaled_summary["split"] = pd.Categorical(split_scaled_summary["split"], categories=split_order, ordered=True)
split_scaled_summary = split_scaled_summary.sort_values("split").reset_index(drop=True)

split_scaled_summary


,split,mean,std,min,max
0,train,0.000,1.000,-2.528,9.451
1,validation,-0.165,1.028,-2.045,16.043
2,test,-0.344,1.249,-2.528,14.998


In [142]:
# Drop helper column used only for split design (keep final schema consistent across cases)
df = df.drop(columns=["date"], errors="ignore")
df.columns


Index(['time', 'value', 'is_anomaly', 'case_study', 'hour_of_day',
       'day_of_week', 'is_weekend', 'split', 'value_scaled'],
      dtype='object')

## Reorder columns for a clean schema

In [144]:
column_order = [
    "time",
    "value",
    "value_scaled",
    "is_anomaly",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "split",
    "case_study",
]

# Reorder the dataframe using the defined schema
df = df[column_order]

# Quick check of the first few rows
df.head()



,time,value,value_scaled,is_anomaly,hour_of_day,day_of_week,is_weekend,split,case_study
0,2017-04-30 16:00:00,1.666667,-0.919925,0,16,6,1,train,aiops_kpi
1,2017-04-30 16:01:00,1.833333,-0.759141,0,16,6,1,train,aiops_kpi
2,2017-04-30 16:02:00,1.750000,-0.839533,0,16,6,1,train,aiops_kpi
3,2017-04-30 16:03:00,2.000000,-0.598357,0,16,6,1,train,aiops_kpi
4,2017-04-30 16:04:00,1.916667,-0.678749,0,16,6,1,train,aiops_kpi


## Recreate split-specific dataframes so they include `value_scaled`

In [147]:
train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()


In [148]:
#Quick check, rows should be equal
len(df), len(train_df) + len(validation_df) + len(test_df)

(129035, 129035)

## Step 10 – Save processed CPU utilisation datasets to disk

In [157]:
import os

# Folder for processed AIOps KPI data 
output_dir = "../data/processed/aiops_kpi"
os.makedirs(output_dir, exist_ok=True)

# Save full processed dataframe (ground truth processed file)
full_path = os.path.join(output_dir, "aiops_kpi_full.csv")
df.to_csv(full_path, index=False)

# File paths for the three splits
train_path = os.path.join(output_dir, "aiops_kpi_train.csv")
val_path   = os.path.join(output_dir, "aiops_kpi_validation.csv")
test_path  = os.path.join(output_dir, "aiops_kpi_test.csv")

# Save each split without the index column
train_df.to_csv(train_path, index=False)
validation_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

train_path, val_path, test_path, full_path


('../data/processed/aiops_kpi/aiops_kpi_train.csv',
 '../data/processed/aiops_kpi/aiops_kpi_validation.csv',
 '../data/processed/aiops_kpi/aiops_kpi_test.csv',
 '../data/processed/aiops_kpi/aiops_kpi_full.csv')

### Save processed AIOps KPI datasets to disk

The processed AIOps KPI outputs are saved to the `data/processed/aiops_kpi/` folder as four CSV files:

- **`aiops_kpi_full.csv`**
  - Contains the complete Case D KPI time series after preprocessing.
  - Includes the standardised cross-case schema used across all case studies:
    - `time, value, value_scaled, is_anomaly, hour_of_day, day_of_week, is_weekend, split, case_study`
  - Serves as the canonical processed reference file for Case D and the source for all split-specific files.

- **`aiops_kpi_train.csv`**
  - Subset of `aiops_kpi_full.csv` where `split == "train"`.
  - Used as the fitting-only segment for preprocessing statistics (e.g., StandardScaler parameters) and as the training data for models.

- **`aiops_kpi_validation.csv`**
  - Subset of `aiops_kpi_full.csv` where `split == "validation"`.
  - Used for model selection and tuning under a pre-incident lead-up regime while maintaining strict chronology.

- **`aiops_kpi_test.csv`**
  - Subset of `aiops_kpi_full.csv` where `split == "test"`.
  - Held-out segment used for final evaluation, containing the high-anomaly incident window and post-incident behaviour.

All split-specific files are derived directly from the single processed full file, ensuring a consistent and reproducible link between the canonical dataset and the train/validation/test segments used in downstream experiments.
